In [1]:
# C12 Step 1 — Load and Verify Datasets

import pandas as pd
import numpy as np

# Load datasets
energy_c12 = pd.read_csv(
    "../../Feature_engineered_datasets/energy_feature_engineered.csv"
)

weather_c12 = pd.read_csv(
    "../../Feature_engineered_datasets/weather_feature_engineered.csv"
)

air_c12 = pd.read_csv(
    "../../Feature_engineered_datasets/air_quality_feature_engineered.csv"
)

print("C12 — Dataset Verification")
print("=" * 60)

print("\nEnergy:")
print("Shape:", energy_c12.shape)
print("Columns:", list(energy_c12.columns))

print("\nWeather:")
print("Shape:", weather_c12.shape)
print("Columns:", list(weather_c12.columns))

print("\nAir Quality:")
print("Shape:", air_c12.shape)
print("Columns:", list(air_c12.columns))

C12 — Dataset Verification

Energy:
Shape: (1000, 19)
Columns: ['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']

Weather:
Shape: (96429, 18)
Columns: ['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Pressure (millibars)', 'Daily Summary', 'Year', 'Month', 'Hour', 'DayOfWeek', 'Season', 'Temperature_Difference', 'Wind_Chill_Indicator']

Air Quality:
Shape: (500, 19)
Columns: ['PM2.5', 'PM10', 'NOx', 'NO2', 'SO2', 'VOCs', 'CO', 'CO2', 'CH4', 'Temperature', 'Humidity', 'Wind_Direction', 'Location_Type', 'Source_Label', 'PM_Ratio', 'NOx_NO2_Ratio', 'CO_CO2_Ratio', 'Pollution_Score', 'Pollution_Level']


In [2]:
# C12 Step 2 — Create Weather Month + Hour Context

weather_context_c12 = (
    weather_c12
    .groupby(["Month", "Hour"])[
        [
            "Temperature (C)",
            "Apparent Temperature (C)",
            "Humidity",
            "Wind Speed (km/h)",
            "Visibility (km)",
            "Pressure (millibars)"
        ]
    ]
    .mean()
    .reset_index()
)

print("C12 — Weather Context")
print("=" * 60)

print("Shape:", weather_context_c12.shape)

print("\nFirst 10 rows:")
print(weather_context_c12.head(10))

print("\nMissing values:")
print(weather_context_c12.isnull().sum())

C12 — Weather Context
Shape: (288, 8)

First 10 rows:
   Month  Hour  Temperature (C)  Apparent Temperature (C)  Humidity  \
0      1     0        -0.291105                 -2.963718  0.883607   
1      1     1        -0.410427                 -3.252786  0.881056   
2      1     2        -0.566846                 -3.283399  0.883959   
3      1     3        -0.614614                 -3.371717  0.889795   
4      1     4        -0.772988                 -3.598110  0.886041   
5      1     5        -0.790893                 -3.610101  0.887947   
6      1     6        -0.761518                 -3.637341  0.890528   
7      1     7        -0.457592                 -3.307739  0.884839   
8      1     8         0.391544                 -2.569762  0.864516   
9      1     9         1.311274                 -1.705279  0.841173   

   Wind Speed (km/h)  Visibility (km)  Pressure (millibars)  
0          10.346729         8.064353           1019.761290  
1          10.985346         8.196175   

In [3]:
# C12 Step 3 — Merge Weather Context with Energy

energy_fusion_c12 = energy_c12.merge(
    weather_context_c12,
    on=["Month", "Hour"],
    how="left"
)

print("C12 — Energy + Weather Fusion")
print("=" * 60)

print("Shape:", energy_fusion_c12.shape)

print("\nMissing values after merge:")
print(
    energy_fusion_c12[
        [
            "Temperature (C)",
            "Apparent Temperature (C)",
            "Humidity_y",
            "Wind Speed (km/h)",
            "Visibility (km)",
            "Pressure (millibars)"
        ]
    ].isnull().sum()
)

print("\nFirst 5 rows:")
print(
    energy_fusion_c12[
        [
            "Month",
            "Hour",
            "Temperature",
            "EnergyConsumption",
            "Temperature (C)",
            "Apparent Temperature (C)",
            "Humidity_y",
            "Wind Speed (km/h)",
            "Visibility (km)",
            "Pressure (millibars)"
        ]
    ].head()
)

C12 — Energy + Weather Fusion
Shape: (1000, 25)

Missing values after merge:
Temperature (C)             0
Apparent Temperature (C)    0
Humidity_y                  0
Wind Speed (km/h)           0
Visibility (km)             0
Pressure (millibars)        0
dtype: int64

First 5 rows:
   Month  Hour  Temperature  EnergyConsumption  Temperature (C)  \
0      1     0    25.139433          75.364373        -0.291105   
1      1     1    27.731651          83.401855        -0.410427   
2      1     2    28.704277          78.270888        -0.566846   
3      1     3    20.080469          56.519850        -0.614614   
4      1     4    23.097359          70.811732        -0.772988   

   Apparent Temperature (C)  Humidity_y  Wind Speed (km/h)  Visibility (km)  \
0                 -2.963718    0.883607          10.346729         8.064353   
1                 -3.252786    0.881056          10.985346         8.196175   
2                 -3.283399    0.883959          10.520854         8.157979

In [4]:
# C12 Step 4 — Create Leakage-Safe Feature Set

energy_features_c12 = [
    "Temperature",
    "Humidity_x",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status"
]

weather_features_c12 = [
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity_y",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)"
]

feature_columns_c12 = energy_features_c12 + weather_features_c12

X_c12 = energy_fusion_c12[feature_columns_c12].copy()
y_c12 = energy_fusion_c12["EnergyConsumption"].copy()

print("C12 — Leakage-Safe Feature Set")
print("=" * 60)

print("X shape:", X_c12.shape)
print("y shape:", y_c12.shape)

print("\nFeatures:")
for i, col in enumerate(feature_columns_c12, 1):
    print(f"{i}. {col}")

print("\nTarget:")
print("EnergyConsumption")

print("\nMissing values:")
print(X_c12.isnull().sum().sum())

print("\nLeakage columns excluded:")
print([
    "Energy_Per_Occupant",
    "Renewable_Energy_Ratio",
    "Energy_Per_SquareFoot"
])

C12 — Leakage-Safe Feature Set
X shape: (1000, 20)
y shape: (1000,)

Features:
1. Temperature
2. Humidity_x
3. SquareFootage
4. Occupancy
5. HVACUsage
6. LightingUsage
7. RenewableEnergy
8. DayOfWeek
9. Holiday
10. Hour
11. Month
12. Weekend
13. HVAC_Status
14. Lighting_Status
15. Temperature (C)
16. Apparent Temperature (C)
17. Humidity_y
18. Wind Speed (km/h)
19. Visibility (km)
20. Pressure (millibars)

Target:
EnergyConsumption

Missing values:
0

Leakage columns excluded:
['Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']


In [5]:
# C12 Step 5 — Preprocessing and Train/Test Split

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Categorical and numerical features
categorical_features_c12 = [
    "HVACUsage",
    "LightingUsage",
    "DayOfWeek",
    "Holiday",
    "Weekend"
]

numerical_features_c12 = [
    col for col in feature_columns_c12
    if col not in categorical_features_c12
]

# Train-test split
X_train_c12, X_test_c12, y_train_c12, y_test_c12 = train_test_split(
    X_c12,
    y_c12,
    test_size=0.20,
    random_state=42
)

# Preprocessor
preprocessor_c12 = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", drop="first"),
            categorical_features_c12
        ),
        (
            "num",
            "passthrough",
            numerical_features_c12
        )
    ]
)

# Fit only on training data
X_train_c12_encoded = preprocessor_c12.fit_transform(X_train_c12)

# Transform test data
X_test_c12_encoded = preprocessor_c12.transform(X_test_c12)

print("C12 — Preprocessing & Split")
print("=" * 60)

print("X_train shape:", X_train_c12.shape)
print("X_test shape :", X_test_c12.shape)
print("y_train shape:", y_train_c12.shape)
print("y_test shape :", y_test_c12.shape)

print("\nEncoded shapes:")
print("X_train encoded:", X_train_c12_encoded.shape)
print("X_test encoded :", X_test_c12_encoded.shape)

print("\nCategorical features:")
print(categorical_features_c12)

print("\nNumerical features:")
print(numerical_features_c12)

C12 — Preprocessing & Split
X_train shape: (800, 20)
X_test shape : (200, 20)
y_train shape: (800,)
y_test shape : (200,)

Encoded shapes:
X_train encoded: (800, 25)
X_test encoded : (200, 25)

Categorical features:
['HVACUsage', 'LightingUsage', 'DayOfWeek', 'Holiday', 'Weekend']

Numerical features:
['Temperature', 'Humidity_x', 'SquareFootage', 'Occupancy', 'RenewableEnergy', 'Hour', 'Month', 'HVAC_Status', 'Lighting_Status', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity_y', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']


In [6]:
# C12 Step 6 — Train Fusion Baseline Models

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Linear Regression
lr_c12 = LinearRegression()
lr_c12.fit(X_train_c12_encoded, y_train_c12)
y_pred_lr_c12 = lr_c12.predict(X_test_c12_encoded)

# Decision Tree
dt_c12 = DecisionTreeRegressor(
    random_state=42
)
dt_c12.fit(X_train_c12_encoded, y_train_c12)
y_pred_dt_c12 = dt_c12.predict(X_test_c12_encoded)

# Random Forest
rf_c12 = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_c12.fit(X_train_c12_encoded, y_train_c12)
y_pred_rf_c12 = rf_c12.predict(X_test_c12_encoded)

print("C12 — Fusion Models Trained")
print("=" * 60)

print("Linear Regression predictions:", len(y_pred_lr_c12))
print("Decision Tree predictions   :", len(y_pred_dt_c12))
print("Random Forest predictions   :", len(y_pred_rf_c12))

C12 — Fusion Models Trained
Linear Regression predictions: 200
Decision Tree predictions   : 200
Random Forest predictions   : 200


In [7]:
# C12 Step 7 — Evaluate Fusion Models

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score
)
import numpy as np

def evaluate_c12_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    # Symmetric MAPE
    smape = np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # Normalized RMSE
    nrmse = (rmse / np.mean(y_true)) * 100

    median_ae = median_absolute_error(y_true, y_pred)
    explained_var = explained_variance_score(y_true, y_pred)

    return [
        mae,
        mse,
        rmse,
        r2,
        smape,
        nrmse,
        median_ae,
        explained_var
    ]


results_c12 = []

models_c12 = {
    "Fusion Linear Regression": y_pred_lr_c12,
    "Fusion Decision Tree": y_pred_dt_c12,
    "Fusion Random Forest": y_pred_rf_c12
}

for model_name, predictions in models_c12.items():

    metrics = evaluate_c12_model(
        y_test_c12,
        predictions
    )

    results_c12.append([
        model_name,
        *metrics
    ])


results_c12_df = pd.DataFrame(
    results_c12,
    columns=[
        "Model",
        "MAE",
        "MSE",
        "RMSE",
        "R2",
        "sMAPE (%)",
        "NRMSE (%)",
        "Median AE",
        "Explained Variance"
    ]
)

print("C12 — Fusion Model Results")
print("=" * 80)

print(
    results_c12_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

C12 — Fusion Model Results
                   Model    MAE     MSE   RMSE     R2  sMAPE (%)  NRMSE (%)  Median AE  Explained Variance
Fusion Linear Regression 4.2118 27.8125 5.2738 0.5754     5.5262     6.8418     3.6662              0.5754
    Fusion Decision Tree 5.6963 52.9412 7.2761 0.1917     7.4418     9.4394     4.9848              0.1945
    Fusion Random Forest 4.5268 31.3144 5.5959 0.5219     5.9151     7.2597     4.2050              0.5219


In [8]:
# C12 Step 8 — Controlled Leakage Check
# IMPORTANT: This is ONLY a diagnostic experiment.
# Renewable_Energy_Ratio must NOT be used in the final model.

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# ---------------------------------------------------------
# 1. Create diagnostic feature set
# ---------------------------------------------------------

leaky_features_c12 = [
    "Temperature",
    "Humidity_x",
    "SquareFootage",
    "Occupancy",
    "HVACUsage",
    "LightingUsage",
    "RenewableEnergy",
    "DayOfWeek",
    "Holiday",
    "Hour",
    "Month",
    "Weekend",
    "HVAC_Status",
    "Lighting_Status",
    "Temperature (C)",
    "Apparent Temperature (C)",
    "Humidity_y",
    "Wind Speed (km/h)",
    "Visibility (km)",
    "Pressure (millibars)",
    "Renewable_Energy_Ratio"   # LEAKAGE — diagnostic only
]

X_c12_leaky = energy_fusion_c12[leaky_features_c12].copy()
y_c12_leaky = energy_fusion_c12["EnergyConsumption"].copy()

# ---------------------------------------------------------
# 2. Same 80/20 split
# ---------------------------------------------------------

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_c12_leaky,
    y_c12_leaky,
    test_size=0.20,
    random_state=42
)

# ---------------------------------------------------------
# 3. Preprocessing
# ---------------------------------------------------------

categorical_leaky_c12 = [
    "HVACUsage",
    "LightingUsage",
    "DayOfWeek",
    "Holiday",
    "Weekend"
]

numerical_leaky_c12 = [
    col for col in leaky_features_c12
    if col not in categorical_leaky_c12
]

preprocessor_leaky_c12 = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_leaky_c12
        ),
        (
            "num",
            "passthrough",
            numerical_leaky_c12
        )
    ]
)

X_train_leaky_encoded = preprocessor_leaky_c12.fit_transform(
    X_train_leaky
)

X_test_leaky_encoded = preprocessor_leaky_c12.transform(
    X_test_leaky
)

# ---------------------------------------------------------
# 4. Train Linear Regression
# ---------------------------------------------------------

lr_leaky_c12 = LinearRegression()

lr_leaky_c12.fit(
    X_train_leaky_encoded,
    y_train_leaky
)

y_pred_leaky_c12 = lr_leaky_c12.predict(
    X_test_leaky_encoded
)

# ---------------------------------------------------------
# 5. Evaluate
# ---------------------------------------------------------

mae_leaky = mean_absolute_error(
    y_test_leaky,
    y_pred_leaky_c12
)

mse_leaky = mean_squared_error(
    y_test_leaky,
    y_pred_leaky_c12
)

rmse_leaky = np.sqrt(mse_leaky)

r2_leaky = r2_score(
    y_test_leaky,
    y_pred_leaky_c12
)

print("C12 — Controlled Leakage Check")
print("=" * 60)

print("Features:", X_c12_leaky.shape[1])
print("Encoded features:", X_train_leaky_encoded.shape[1])

print("\nLinear Regression WITH Renewable_Energy_Ratio")
print("MAE :", round(mae_leaky, 4))
print("MSE :", round(mse_leaky, 4))
print("RMSE:", round(rmse_leaky, 4))
print("R²  :", round(r2_leaky, 4))

print("\nComparison")
print("-" * 60)
print("Leakage-safe C12 LR : RMSE = 5.2738 | R² = 0.5754")
print(
    f"Leaky diagnostic LR : RMSE = {rmse_leaky:.4f} | R² = {r2_leaky:.4f}"
)

print("\nIMPORTANT:")
print("Renewable_Energy_Ratio is target-derived and MUST NOT be used")
print("in the final predictive model.")

C12 — Controlled Leakage Check
Features: 21
Encoded features: 26

Linear Regression WITH Renewable_Energy_Ratio
MAE : 2.7282
MSE : 12.5651
RMSE: 3.5447
R²  : 0.8082

Comparison
------------------------------------------------------------
Leakage-safe C12 LR : RMSE = 5.2738 | R² = 0.5754
Leaky diagnostic LR : RMSE = 3.5447 | R² = 0.8082

IMPORTANT:
Renewable_Energy_Ratio is target-derived and MUST NOT be used
in the final predictive model.
